# Android heading convention and stride-gain diagnostic

## tl;dr

- B1 v1.1.0 now follows Android `SensorManager.getOrientation()` azimuth semantics.
- Correct semantics lower platform heading MAE to about 66.5–66.8°, but mirror and catastrophic turn failures remain.
- A predeclared stride-gain sweep shows that distance is calibration-sensitive; no gain is selected from this test sequence.
- The current step detector is also rate-sensitive because identical data produces materially different distance at 50 and 100 Hz.
- Decision: keep Stop, do not start a personal pilot, and fix body-heading and rate stability before calibration.

## Method and claim boundary

The estimator receives only raw accelerometer, gyroscope, Game Rotation Vector, and sensor timestamps. Android azimuth is computed as `atan2(R[1], R[4])`, then converted to mathematical heading using `pi/2 - azimuth`.

Stride uses `K * amplitude ** 0.25` for predeclared `K = 0.364, 0.400, 0.450, 0.640`. Ground truth is evaluated only after every output exists. These records are a sensitivity analysis, not hyperparameter selection.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

report_path = Path("/outputs/ronin-a054_1-heading-stride.json")
sequence_root = Path("/data/ronin/a054_1")
if not report_path.exists():
    subprocess.run(
        [
            sys.executable,
            "research/pdr/scripts/analyze_heading_stride.py",
            "--sequence-root", str(sequence_root),
            "--output", str(report_path),
        ],
        check=True,
    )
report = json.loads(report_path.read_text(encoding="utf-8"))
print(
    f"records={report['record_count']} "
    f"future_violations={report['future_sample_violations']}"
)
print(report["decision"])

records=16 future_violations=0
{'heading': 'stop-device-orientation-as-body-heading', 'stride': 'sensitivity-only-no-test-sequence-selection', 'personal_pilot': 'not-authorized-by-this-evidence'}


## Results

In [2]:
print(f"{'rate':>4} {'profile':<16} {'K':>6} {'distance':>10} {'scale':>8} {'heading':>9} {'turn':>7} {'mirror':>7}")
print("-" * 82)
for record in report["records"]:
    metrics = record["metrics"]
    print(
        f"{record['target_rate_hz']:>4} {record['capability_profile']:<16} "
        f"{record['weinberg_gain']:>6.3f} {metrics['estimated_distance_m']:>10.1f} "
        f"{metrics['distance_scale_error']:>8.3f} {metrics['heading_mae_deg']:>9.1f} "
        f"{metrics['turn_angle_mae_deg']:>7.1f} {str(metrics['mirrored']):>7}"
    )

rate profile               K   distance    scale   heading    turn  mirror
----------------------------------------------------------------------------------
  50 imu6              0.364      406.9    0.067      78.4    44.3   False
  50 imu6              0.400      439.2    0.152      78.4    44.3   False
  50 imu6              0.450      477.4    0.252      78.4    44.3   False
  50 imu6              0.640      570.2    0.496      78.4    44.3   False
  50 platform-fused    0.364      406.9    0.067      66.8    60.4    True
  50 platform-fused    0.400      439.2    0.152      66.8    60.4    True
  50 platform-fused    0.450      477.4    0.252      66.8    60.4    True
  50 platform-fused    0.640      570.2    0.496      66.8    60.4    True
 100 imu6              0.364      432.1    0.133      77.0    43.5   False
 100 imu6              0.400      466.2    0.223      77.0    43.5   False
 100 imu6              0.450      508.6    0.334      77.0    43.5   False
 100 imu6        

## Reproducibility and leakage assertions

In [3]:
assert report["record_count"] == 16
assert report["future_sample_violations"] == 0
assert report["stride_sensitivity"]["gains"] == [0.364, 0.4, 0.45, 0.64]
assert "do not select" in report["stride_sensitivity"]["selection_rule"]
assert report["decision"]["heading"] == "stop-device-orientation-as-body-heading"
assert report["decision"]["personal_pilot"] == "not-authorized-by-this-evidence"

for rate in (50, 100):
    for profile in ("imu6", "platform-fused"):
        group = [
            record for record in report["records"]
            if record["target_rate_hz"] == rate
            and record["capability_profile"] == profile
        ]
        distances = [record["metrics"]["estimated_distance_m"] for record in group]
        headings = {round(record["metrics"]["heading_mae_deg"], 12) for record in group}
        assert distances == sorted(distances)
        assert len(headings) == 1
        assert all(
            record["metrics"]["future_sample_violations"] == 0 for record in group
        )

print("All causality, predeclared-sweep, monotonic-distance, and no-selection assertions passed.")

All causality, predeclared-sweep, monotonic-distance, and no-selection assertions passed.


## Takeaways

1. Android coordinate semantics are now explicit and tested, but device heading remains distinct from body heading.
2. The low distance error at one fixed coefficient is not a valid selection result because this is the test sequence and the coefficient is placement/person dependent.
3. Distance disagreement between 50 and 100 Hz must be addressed before calibration.
4. Public-sequence performance and Android lifecycle feasibility remain separate decisions.